# 02 — S01 provenance contract and end-to-end tutorial

**Question:** Can namespaced records be aligned without ambiguity, can a complete synthetic signal-to-summary workflow run reproducibly, and can changed lineage fail closed?

**Permitted inputs:** 24 mathematical signal pairs generated in memory. QC measurements and quality variables are synthetic interface fixtures, not inferred measures of signal realism. No participant-level material or E01 findings are consumed.

**Outputs:** `02_fixture_protocol.json`, `02_tutorial_summary.json`, `02_provenance_failures.json`, and `02_execution_manifest.json` under `notebooks/artifacts/`.

**Seeds:** fixture 20260918; model/split 1201. **Claims:** P0.3 identity/lineage consistency and P0.4 tutorial integration only. Predictive profiles demonstrate APIs; they are not notebook 03's comparator benchmark.

Clean setup and independent verification commands are in `PROVENANCE_TUTORIAL.md`.

In [1]:
from pathlib import Path
import importlib.util
import json
from dataclasses import replace
from neurofusion_qa.provenance import (ObservationIdentity, checked_join, canonical_json_sha256, verify_execution_manifest)
START = Path.cwd().resolve()
ROOT = next(p for p in (START, *START.parents) if (p / 'src/neurofusion_qa').is_dir() and (p / 'pyproject.toml').is_file())
spec = importlib.util.spec_from_file_location('s01_provenance_tutorial', ROOT / 'scripts/s01_provenance_tutorial.py')
tutorial = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tutorial)
ARTIFACTS = ROOT / 'notebooks/artifacts'
tutorial.CONFIGURATION

{'scope': 'synthetic_software_tutorial_not_benchmark',
 'fixture_seed': 20260918,
 'model_seed': 1201,
 'records': 24,
 'eeg_rate_hz': 250.0,
 'fmri_rate_hz': 1.25,
 'eeg_samples': 750,
 'fmri_samples': 128,
 'held_out_fraction': 0.3333333333333333,
 'inner_folds': 3,
 'fixed_c': 0.1,
 'quality_directions': [1, -1, 1, -1],
 'threshold_source': 'training_only_oof_predictions'}

## Explicit identities and checked alignment

An observation key retains namespace, subject, session, and run. Modality is explicit but excluded from the cross-modality pairing key. Reordering is allowed and resolved by returned indices; missing rows and undeclared cardinality are not silently accepted. Identical local record IDs in two namespaces remain distinct.

In [2]:
first = ObservationIdentity('math-a', 'record-001', 'session-01', 'run-01', 'eeg')
second = replace(first, namespace='math-b')
left = [first, second]
right = [replace(second, modality='fmri'), replace(first, modality='fmri')]
pairs = checked_join(left, right, cardinality='one_to_one')
assert pairs == [(0, 1), (1, 0)]
assert first.observation_key != second.observation_key
{'matched_indices': pairs, 'namespace_preserved': True}

{'matched_indices': [(0, 1), (1, 0)], 'namespace_preserved': True}

## Fail-closed provenance conditions

Trigger duplicate and missing identities, cross-namespace mismatches, forbidden many-to-many joins, reordered snapshots, altered configurations, missing derivatives, modified input files, tampered digests, and stale outputs. Temporary files contain only synthetic JSON and are isolated from project evidence. Every case must raise its specific diagnostic.

In [3]:
failures = tutorial.provenance_failures()
assert len(failures) == 11
assert all(row['exception'] == 'ValueError' for row in failures)
failures

[{'case': 'duplicate_identity',
  'exception': 'ValueError',
  'message': 'duplicate left identity'},
 {'case': 'missing_identity',
  'exception': 'ValueError',
  'message': 'missing or unexpected observation keys; join would drop records'},
 {'case': 'namespace_collision',
  'exception': 'ValueError',
  'message': 'cross-namespace collision in unmatched observation keys'},
 {'case': 'many_to_many',
  'exception': 'ValueError',
  'message': 'unsupported join cardinality; many-to-many joins are forbidden'},
 {'case': 'reordered_inputs',
  'exception': 'ValueError',
  'message': 'manifest differs from the trusted snapshot digest'},
 {'case': 'altered_configuration',
  'exception': 'ValueError',
  'message': 'configuration differs from the expected configuration'},
 {'case': 'tampered_manifest',
  'exception': 'ValueError',
  'message': 'manifest content hash mismatch'},
 {'case': 'tampered_output_hash',
  'exception': 'ValueError',
  'message': 'outputs artifact hash mismatch (changed or

## Complete synthetic signal-to-summary workflow

Extract EEG-33 and fMRI-1440 from every generated signal pair, verify computational integrity, align a deliberately shuffled modality table, and use one shared stratified train/test split. Fit imputers, model scaling, and quality scaling only on the 16 training records. The fixed classifier setting is declared before execution; training-only three-fold predictions select each modality's decision threshold.

One held-out QC fixture fails censored-fraction criteria while its features retain computational integrity. Its fMRI availability and quality-aware weight become zero, recovering the EEG prediction. The equal-fusion branch intentionally ignores the QC fixture to demonstrate the separate fusion interface; it is not a scientifically admissible pipeline comparison.

A separate constant feature vector fails integrity. This does not make a scientific QC statement. Aggregate evaluation profiles are interface demonstrations on eight generated held-out records, not estimates of clinical validity or method superiority. No signal arrays or individual predictions are written.

In [4]:
summary = tutorial.run_signal_tutorial()
assert summary['eeg_shape'] == [24, 33] and summary['fmri_shape'] == [24, 1440]
assert summary['training_records'] == 16 and summary['held_out_records'] == 8
assert summary['train_test_disjoint'] and summary['quality_fit_on_training_only']
assert summary['qc_rejected_modality_receives_zero_weight']
assert not summary['separate_integrity_failure']['passed']
assert not summary['qc_failure_with_intact_features']['passed']
{key: summary[key] for key in ('records', 'eeg_shape', 'fmri_shape', 'model_reports', 'scientific_qc_fixture_failures', 'profiles')}

{'records': 24,
 'eeg_shape': [24, 33],
 'fmri_shape': [24, 1440],
 'model_reports': {'eeg': {'fixed_c': 0.1,
   'training_oof_threshold': 0.4099999999999998,
   'features': 33,
   'fit_records': 16},
  'fmri': {'fixed_c': 0.1,
   'training_oof_threshold': 0.6749999999999997,
   'features': 1440,
   'fit_records': 16}},
 'scientific_qc_fixture_failures': 1,
 'profiles': {'eeg': {'roc_auc': 1.0,
   'average_precision': 1.0,
   'brier_score': 0.015456810150918978,
   'log_loss': 0.11925283610580809},
  'fmri': {'roc_auc': 1.0,
   'average_precision': 1.0,
   'brier_score': 0.12172610308581024,
   'log_loss': 0.40706416470633733},
  'equal': {'roc_auc': 1.0,
   'average_precision': 1.0,
   'brier_score': 0.054722508101775445,
   'log_loss': 0.24515627123820544},
  'quality': {'roc_auc': 1.0,
   'average_precision': 1.0,
   'brier_score': 0.05102333360725732,
   'log_loss': 0.2268623642554352}}}

## Same-environment deterministic replay

Regenerate the signals and refit the complete workflow. Equality of canonical summary hashes checks reproducibility in this environment, not bitwise portability across all platforms or dependency versions.

In [5]:
replay = tutorial.run_signal_tutorial()
assert canonical_json_sha256(summary) == canonical_json_sha256(replay)
{'same_seed_replay_matches': True, 'summary_sha256': canonical_json_sha256(summary)}

{'same_seed_replay_matches': True,
 'summary_sha256': 'cd5ea14908bb43fc8d57b0ba8cdf477b817d523ff5d8a0d1db52940ce8b38204'}

## Bind and independently verify the evidence

The manifest binds ordered source/protocol inputs, configuration, source-snapshot revision, dependency versions, seed, and output-file hashes. It is serialized, loaded from disk, and independently verified against the current files and expected configuration. Source changes invalidate the saved snapshot until it is deliberately regenerated.

A hash is not a signature. Without an independently trusted digest, a party can rewrite both payload and checksum. The manifest proves consistency with recorded bytes, not that a process ran, that identity labels are true, or that results are scientifically valid. No external timestamps or clinical claims are implied.

In [6]:
evidence = tutorial.write_evidence(summary, failures)
manifest = json.loads((ROOT / evidence['path']).read_text(encoding='utf-8'))
verified = verify_execution_manifest(manifest, ROOT, expected_configuration=tutorial.CONFIGURATION, expected_manifest_sha256=evidence['manifest_sha256'])
assert len(verified['outputs']) == 2
assert summary['e01_data_accessed'] is False and summary['clinical_claim_supported'] is False
evidence

{'path': 'notebooks/artifacts/02_execution_manifest.json',
 'manifest_sha256': '3a1479cb858a5b663819ddcaec077286ab8c103ff4d8ea56017356d433357800',
 'outputs_verified': 2}

## Evidence-bound figure exports

Regenerate relevant figures from this execution's evidence. Every family has editable SVG/PDF and 1000 dpi Times New Roman PNG exports, a caption, and source hashes. [Full gallery](figures/README.md). Synthetic software evidence only.

In [7]:
import sys
from IPython.display import Markdown, display
if str(ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(ROOT / 'scripts'))
from s01_evidence_figures import generate
figure_records = generate('02')
for item in figure_records:
    display(Markdown(f"### {item['id']}\n\n![{item['id']}](figures/previews/{item['id']}.png)\n\n{item['caption']}"))
{'figure_sets': len(figure_records), 'export_formats': ['png', 'svg', 'pdf']}

### F04_provenance_lineage

![F04_provenance_lineage](figures/previews/F04_provenance_lineage.png)

Conceptual flow of the executed tutorial's identity and manifest contract; actual failure checks accompany the schematic. A manifest is not an authenticated signature.

### F05_tutorial_accounting

![F05_tutorial_accounting](figures/previews/F05_tutorial_accounting.png)

Tutorial accounting from executed evidence: disjoint 16/8 split, actual feature shapes, separate QC/integrity checks and fail-closed provenance demonstrations. Small, deliberately easy signals cannot establish prediction performance.

{'figure_sets': 2, 'export_formats': ['png', 'svg', 'pdf']}

Exported F04_provenance_lineage: Times New Roman, 1000 dpi
Exported F05_tutorial_accounting: Times New Roman, 1000 dpi
